# مسیر ترکیبی — رابطه‌ی سالمندی جمعیت و هزینه‌ی سلامت

**منبع داده:** World Bank Open Data
- `SP.POP.65UP.TO.ZS` — Population ages 65 and above (% of total population)
- `SH.XPD.CHEX.PC.CD` — Current health expenditure per capita (current US$)

این نوت‌بوک شامل مراحل زیر است:
1. لود کردن و تبدیل داده از فرمت Wide به Long
2. پاک‌سازی نهایی و ادغام دو شاخص
3. نمودار پراکندگی با خط روند و هایلایت ژاپن
4. تحلیل کمّی وضعیت ژاپن نسبت به خط روند و کشورهای هم‌سطح


## فاز ۱: لود کردن و تبدیل داده از فرمت Wide به Long

In [ ]:
import pandas as pd

# --- لیست ۲۳ کشور انتخابی با کد ISO3 ---
countries = {
    "JPN": "Japan", "KOR": "South Korea", "DEU": "Germany", "ITA": "Italy",
    "FRA": "France", "ESP": "Spain", "SWE": "Sweden", "GBR": "United Kingdom",
    "USA": "United States", "CAN": "Canada", "CHN": "China", "IND": "India",
    "IRN": "Iran", "TUR": "Turkey", "RUS": "Russia", "BRA": "Brazil",
    "MEX": "Mexico", "AUS": "Australia", "POL": "Poland", "PRT": "Portugal",
    "EGY": "Egypt", "NGA": "Nigeria", "ZAF": "South Africa"
}
print("تعداد کشورها:", len(countries))

In [ ]:
# --- لود کردن دو فایل (شیت Data، با رد کردن ۳ ردیف اول متادیتا) ---
pop65_wide = pd.read_excel(
    'Population_ages_65_and_above____of_total_population_.xls',
    sheet_name='Data', header=3, engine='xlrd'
)

health_wide = pd.read_excel(
    'Current_health_expenditure_per_capita.xls',
    sheet_name='Data', header=3, engine='xlrd'
)

print("شکل pop65_wide:", pop65_wide.shape)
print("شکل health_wide:", health_wide.shape)
pop65_wide.head()

In [ ]:
# --- تابعی برای تبدیل هر فایل از Wide به Long و فیلتر کشورهای موردنظر ---
def wide_to_long(df_wide, value_name):
    year_cols = [c for c in df_wide.columns if str(c).isdigit()]
    df_long = df_wide.melt(
        id_vars=['Country Name', 'Country Code'],
        value_vars=year_cols,
        var_name='Year',
        value_name=value_name
    )
    df_long['Year'] = df_long['Year'].astype(int)
    df_long = df_long[df_long['Country Code'].isin(countries)]
    return df_long

pop65_long = wide_to_long(pop65_wide, 'Pop65Plus')
health_long = wide_to_long(health_wide, 'HealthExpPerCapita')

print("شکل pop65_long:", pop65_long.shape)
print("شکل health_long:", health_long.shape)
pop65_long.head()

In [ ]:
# --- بررسی بازه‌ی سال و کشورهای موجود بعد از فیلتر ---
print("بازه‌ی سال Pop65:", pop65_long['Year'].min(), "تا", pop65_long['Year'].max())
print("کشورهای موجود در Pop65:", sorted(pop65_long['Country Code'].unique()))
print()
print("بازه‌ی سال Health:", health_long['Year'].min(), "تا", health_long['Year'].max())
print("کشورهای موجود در Health:", sorted(health_long['Country Code'].unique()))

## فاز ۲: پاک‌سازی نهایی و ادغام دو شاخص

In [ ]:
# --- محدود کردن به بازه‌ی ۲۰۰۰ تا آخرین سال ---
YEAR_MIN = 2000

pop65_clean = pop65_long[pop65_long['Year'] >= YEAR_MIN].copy()
health_clean = health_long[health_long['Year'] >= YEAR_MIN].copy()

print("شکل pop65_clean:", pop65_clean.shape)
print("شکل health_clean:", health_clean.shape)

In [ ]:
# --- بررسی مقادیر گم‌شده در هر دو جدول قبل از ادغام ---
print("مقادیر گم‌شده در pop65_clean:")
print(pop65_clean['Pop65Plus'].isnull().sum())

print("\nمقادیر گم‌شده در health_clean:")
print(health_clean['HealthExpPerCapita'].isnull().sum())

# --- بررسی این‌که آخرین سال دارای داده برای هر کشور کجاست ---
print("\nآخرین سال دارای داده (Pop65) به ازای هر کشور:")
print(pop65_clean.dropna(subset=['Pop65Plus']).groupby('Country Code')['Year'].max().sort_values())

print("\nآخرین سال دارای داده (Health) به ازای هر کشور:")
print(health_clean.dropna(subset=['HealthExpPerCapita']).groupby('Country Code')['Year'].max().sort_values())

In [ ]:
# --- ادغام دو شاخص در یک جدول واحد بر اساس کشور و سال ---
merged = pop65_clean.merge(
    health_clean[['Country Code', 'Year', 'HealthExpPerCapita']],
    on=['Country Code', 'Year'],
    how='inner'
)

print("شکل جدول ادغام‌شده قبل از حذف null:", merged.shape)
merged.head()

In [ ]:
# --- حذف ردیف‌هایی که در هر دو شاخص مقدار ندارن ---
merged_final = merged.dropna(subset=['Pop65Plus', 'HealthExpPerCapita'])

print("شکل نهایی داده (بدون null):", merged_final.shape)
print("\nتعداد ردیف حذف‌شده:", len(merged) - len(merged_final))
print("\nتعداد کشورهای باقی‌مانده:", merged_final['Country Code'].nunique())
print("بازه‌ی سال نهایی:", merged_final['Year'].min(), "تا", merged_final['Year'].max())

In [ ]:
# --- خلاصه آماری نهایی ---
merged_final.describe()

## فاز ۳: نمودار پراکندگی با خط روند و هایلایت ژاپن

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- جدا کردن داده‌ی ژاپن از بقیه ---
japan_data = merged_final[merged_final['Country Code'] == 'JPN'].sort_values('Year')
other_data = merged_final[merged_final['Country Code'] != 'JPN']

print("تعداد نقاط ژاپن:", len(japan_data))
print("تعداد نقاط بقیه کشورها:", len(other_data))

In [ ]:
# --- محاسبه‌ی خط روند ساده (رگرسیون خطی درجه ۱) روی کل داده ---
x = merged_final['Pop65Plus'].values
y = merged_final['HealthExpPerCapita'].values

# --- ضرایب خط روند (شیب و عرض از مبدأ) ---
slope, intercept = np.polyfit(x, y, 1)
trend_x = np.linspace(x.min(), x.max(), 100)
trend_y = slope * trend_x + intercept

print(f"معادله‌ی خط روند: HealthExp = {slope:.2f} × Pop65Plus + {intercept:.2f}")

# --- محاسبه‌ی همبستگی برای اینکه ببینیم رابطه چقدر خطی/قوی هست ---
correlation = np.corrcoef(x, y)[0, 1]
print(f"ضریب همبستگی (r): {correlation:.3f}")

In [ ]:
# --- رسم نمودار پراکندگی نهایی ---
plt.figure(figsize=(11, 7))

# --- نقاط بقیه‌ی کشورها (خاکستری کم‌رنگ) ---
plt.scatter(
    other_data['Pop65Plus'], other_data['HealthExpPerCapita'],
    alpha=0.35, color='gray', s=30, label='Other Countries'
)

# --- خط روند ---
plt.plot(trend_x, trend_y, color='black', linestyle='--', linewidth=2, label='Trend Line')

# --- نقاط ژاپن (هایلایت‌شده و به‌هم‌وصل) ---
plt.plot(
    japan_data['Pop65Plus'], japan_data['HealthExpPerCapita'],
    color='crimson', linewidth=1.5, alpha=0.6, zorder=4
)
plt.scatter(
    japan_data['Pop65Plus'], japan_data['HealthExpPerCapita'],
    color='crimson', s=60, label='Japan', zorder=5, edgecolor='black'
)

# --- علامت‌گذاری آخرین سال ژاپن با یک ستاره ---
last_japan = japan_data.iloc[-1]
plt.scatter(
    last_japan['Pop65Plus'], last_japan['HealthExpPerCapita'],
    color='gold', s=250, marker='*', zorder=6,
    edgecolor='black', label=f"Japan {int(last_japan['Year'])}"
)

plt.title('Population Aging vs. Health Expenditure per Capita (2000-2024)', fontsize=13)
plt.xlabel('Population Ages 65 and Above (% of total)')
plt.ylabel('Current Health Expenditure per Capita (current US$)')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

## فاز ۴: تحلیل کمّی وضعیت ژاپن نسبت به خط روند

In [ ]:
# --- محاسبه‌ی مقدار پیش‌بینی‌شده توسط خط روند برای هر نقطه‌ی ژاپن ---
japan_data = japan_data.copy()
japan_data['Predicted'] = slope * japan_data['Pop65Plus'] + intercept
japan_data['Residual'] = japan_data['HealthExpPerCapita'] - japan_data['Predicted']
japan_data['Residual_Pct'] = (japan_data['Residual'] / japan_data['Predicted']) * 100

print("وضعیت ژاپن نسبت به خط روند در طول زمان:")
print(japan_data[['Year', 'Pop65Plus', 'HealthExpPerCapita', 'Predicted', 'Residual_Pct']].round(1))

In [ ]:
# --- تمرکز ویژه روی آخرین سال موجود ---
last = japan_data.iloc[-1]
print(f"\nدر سال {int(last['Year'])}:")
print(f"درصد جمعیت ۶۵+ ژاپن: {last['Pop65Plus']:.1f}%")
print(f"هزینه‌ی سلامت سرانه‌ی واقعی ژاپن: ${last['HealthExpPerCapita']:.0f}")
print(f"هزینه‌ی سلامت سرانه‌ی پیش‌بینی‌شده توسط خط روند: ${last['Predicted']:.0f}")
print(f"اختلاف: {last['Residual_Pct']:.1f}% {'بیشتر' if last['Residual_Pct']>0 else 'کمتر'} از حد انتظار")

In [ ]:
# --- مقایسه با چند کشور هم‌سطح از نظر سالمندی (آلمان، ایتالیا) در آخرین سال مشترک ---
peers = merged_final[
    (merged_final['Country Code'].isin(['DEU', 'ITA', 'JPN'])) &
    (merged_final['Year'] == last['Year'])
][['Country Name', 'Pop65Plus', 'HealthExpPerCapita']]

peers['Predicted'] = slope * peers['Pop65Plus'] + intercept
peers['Residual_Pct'] = ((peers['HealthExpPerCapita'] - peers['Predicted']) / peers['Predicted']) * 100

print(f"\nمقایسه‌ی ژاپن با کشورهای هم‌سطح در سال {int(last['Year'])}:")
print(peers.round(1))

## جمع‌بندی: پاراگراف تحلیلی نهایی

با وجود اینکه ژاپن بالاترین درصد جمعیت سالمند (۶۵+) در میان ۲۳ کشور بررسی‌شده رو داره (۲۹.۶٪ در سال ۲۰۲۳)، هزینه‌ی سلامت سرانه‌اش به‌طور مداوم **کمتر از حد انتظار** خط روند جهانیه — در آخرین سال، حدود ۴۲٪ پایین‌تر از پیش‌بینی. این الگو یک استثنای موردی نیست؛ در ۲۲ از ۲۴ سال بررسی‌شده (۲۰۰۰-۲۰۲۳) تکرار شده و در سال‌های اخیر (هم‌زمان با شدت‌گرفتن سالمندی) عمیق‌تر هم شده. مقایسه با کشورهای هم‌سطح این تصویر رو روشن‌تر می‌کنه: آلمان با سطح سالمندی پایین‌تر (۲۲.۸٪)، حدود ۳۷٪ **بالاتر** از خط روند هزینه می‌کنه، در حالی که ایتالیا هم مثل ژاپن زیر خط روندست (۳۴.۴٪-). این نشون می‌ده رابطه‌ی سالمندی-هزینه فقط یک قانون طبیعی ثابت نیست، بلکه انتخاب سیاست‌های ملی سلامت نقش تعیین‌کننده‌ای داره. در مورد ژاپن، این وضعیت احتمالاً بازتاب نظام درمانی دولتی و کنترل‌شده‌ایه که با وجود بار سنگین سالمندی، هزینه‌ها رو نسبت به کشورهای هم‌سطح غربی مهار کرده — نکته‌ای که می‌تونه برای کشورهایی که به سمت سالمندی سریع می‌رن (مثل کره‌جنوبی یا حتی ایران در آینده) درس‌آموز باشه.
